### The Agentic Loop

In agent.ipynb, we did function calling by hand. We sent a message and got back a function call. We ran it, sent the result back, and got the answer.

That works for one function call. It breaks down when the model wants to search several times, or when the first search misses the answer. We don't know in advance how many calls the model will want. So we need a loop that keeps calling the model and running tools until it's done. 

An agent is exactly that.

### Anatomy of an agent
With the LLM in the driver's seat, we have an agent. It's an AI assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the developer message. The better the instructions, the better the agent helps.
- Tools, the functions the agent can call to carry out the task. For us that's only search.
- Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

### A developer prompt
So far we've relied on the model to figure out how to search. We make that more reliable with a developer message that spells out how to behave. This is where we give the agent its role. The same message also pushes it toward multiple searches, so we get to watch the loop run more than once.

In [17]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

### A function-call helper
We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.

In [18]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [19]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

The helper returns the exact structure the Responses API expects. When we add more tools later, we'll extend this with more if branches (or switch to a registry).

### Processing one response
Let's process a single model response. We append each output entry to the conversation, print any messages, and run any function calls. Function-call results get appended too.

In [20]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [21]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [22]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [23]:
import json

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration late join FAQ"}
function_call: search {"query":"course discovery can I join after course started enrollment FAQ"}
